In [17]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [18]:
!pip install -q torch torchvision scikit-learn pandas pillow matplotlib

In [19]:
import os
import random
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from torchvision import transforms
from torchvision.models import (
    efficientnet_v2_s,
    EfficientNet_V2_S_Weights,
    convnext_tiny,
    ConvNeXt_Tiny_Weights
)

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix
)

In [20]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)

torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Device: cuda
GPU: Tesla T4


In [5]:
PROJECT_ROOT = Path(
    "/content/drive/MyDrive/Dissertation"
)

IMAGE_ROOT = (
    PROJECT_ROOT
    / "03_Data_Processed"
    / "mammography_512"
)

MANIFEST_FILE = (
    IMAGE_ROOT
    / "breast_level_image_manifest.csv"
)

RESULTS_DIR = (
    PROJECT_ROOT
    / "05_Results"
    / "image_models"
)

MODEL_DIR = (
    PROJECT_ROOT
    / "04_Models"
    / "saved_models"
)

RESULTS_DIR.mkdir(
    parents=True,
    exist_ok=True
)

MODEL_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print("Manifest exists:", MANIFEST_FILE.exists())

Manifest exists: True


In [6]:
df = pd.read_csv(MANIFEST_FILE)

print("Breasts:", len(df))
print("Patients:", df["patient_id"].nunique())

print(df["split"].value_counts())
print(df["target"].value_counts())

Breasts: 2070
Patients: 1361
split
train         1444
test           317
validation     309
Name: count, dtype: int64
target
1    1171
0     899
Name: count, dtype: int64


In [7]:
def create_colab_image_paths(row):

    breast_folder = (
        IMAGE_ROOT
        / row["split"]
        / row["patient_breast_id"]
    )

    return pd.Series({
        "view1_path":
            breast_folder / "view_1.png",

        "view2_path":
            breast_folder / "view_2.png"
    })


path_df = df.apply(
    create_colab_image_paths,
    axis=1
)

df = pd.concat(
    [
        df,
        path_df
    ],
    axis=1
)

missing_view1 = (
    ~df["view1_path"].apply(
        lambda x: Path(x).exists()
    )
).sum()

missing_view2 = (
    ~df["view2_path"].apply(
        lambda x: Path(x).exists()
    )
).sum()

print("Missing View 1:", missing_view1)
print("Missing View 2:", missing_view2)

Missing View 1: 0
Missing View 2: 0


In [8]:
IMAGE_SIZE = 224

train_transform = transforms.Compose([
    transforms.Resize(
        (IMAGE_SIZE, IMAGE_SIZE)
    ),

    transforms.RandomRotation(
        degrees=5
    ),

    transforms.ColorJitter(
        brightness=0.08,
        contrast=0.08
    ),

    transforms.Grayscale(
        num_output_channels=3
    ),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])


eval_transform = transforms.Compose([
    transforms.Resize(
        (IMAGE_SIZE, IMAGE_SIZE)
    ),

    transforms.Grayscale(
        num_output_channels=3
    ),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

In [9]:
class TwoViewMammographyDataset(Dataset):

    def __init__(
        self,
        dataframe,
        transform=None
    ):

        self.df = (
            dataframe
            .reset_index(drop=True)
        )

        self.transform = transform


    def __len__(self):

        return len(self.df)


    def __getitem__(self, idx):

        row = self.df.iloc[idx]

        view1 = Image.open(
            row["view1_path"]
        ).convert("L")

        view2 = Image.open(
            row["view2_path"]
        ).convert("L")


        if self.transform is not None:

            view1 = self.transform(
                view1
            )

            view2 = self.transform(
                view2
            )


        label = torch.tensor(
            int(row["target"]),
            dtype=torch.long
        )

        return (
            view1,
            view2,
            label,
            row["patient_breast_id"]
        )

In [10]:
train_df = df[
    df["split"] == "train"
].copy()

val_df = df[
    df["split"] == "validation"
].copy()

test_df = df[
    df["split"] == "test"
].copy()


train_dataset = TwoViewMammographyDataset(
    train_df,
    transform=train_transform
)

val_dataset = TwoViewMammographyDataset(
    val_df,
    transform=eval_transform
)

test_dataset = TwoViewMammographyDataset(
    test_df,
    transform=eval_transform
)


print(
    len(train_dataset),
    len(val_dataset),
    len(test_dataset)
)

1444 309 317


In [11]:
BATCH_SIZE = 8

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

In [12]:
class TwoViewEfficientNet(nn.Module):

    def __init__(self):

        super().__init__()

        weights = (
            EfficientNet_V2_S_Weights.DEFAULT
        )

        backbone = efficientnet_v2_s(
            weights=weights
        )

        feature_dim = (
            backbone.classifier[1].in_features
        )

        backbone.classifier = nn.Identity()

        self.backbone = backbone

        self.classifier = nn.Sequential(

            nn.Linear(
                feature_dim * 2,
                512
            ),

            nn.ReLU(),

            nn.Dropout(
                0.4
            ),

            nn.Linear(
                512,
                2
            )
        )


    def forward(
        self,
        view1,
        view2
    ):

        features1 = self.backbone(
            view1
        )

        features2 = self.backbone(
            view2
        )

        combined = torch.cat(
            [
                features1,
                features2
            ],
            dim=1
        )

        output = self.classifier(
            combined
        )

        return output

In [13]:
model = TwoViewEfficientNet().to(
    device
)

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-4,
    weight_decay=1e-4
)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="max",
    factor=0.5,
    patience=2
)

print(model.__class__.__name__)

Downloading: "https://download.pytorch.org/models/efficientnet_v2_s-dd5fe13b.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_v2_s-dd5fe13b.pth


100%|██████████| 82.7M/82.7M [00:00<00:00, 153MB/s]


TwoViewEfficientNet


In [14]:
def evaluate_model(
    model,
    data_loader
):

    model.eval()

    all_labels = []
    all_predictions = []
    all_probabilities = []
    all_ids = []

    total_loss = 0

    with torch.no_grad():

        for (
            view1,
            view2,
            labels,
            breast_ids
        ) in data_loader:

            view1 = view1.to(device)
            view2 = view2.to(device)
            labels = labels.to(device)

            outputs = model(
                view1,
                view2
            )

            loss = criterion(
                outputs,
                labels
            )

            total_loss += (
                loss.item()
                * labels.size(0)
            )

            probabilities = (
                torch.softmax(
                    outputs,
                    dim=1
                )[:, 1]
            )

            predictions = (
                torch.argmax(
                    outputs,
                    dim=1
                )
            )

            all_labels.extend(
                labels.cpu().numpy()
            )

            all_predictions.extend(
                predictions.cpu().numpy()
            )

            all_probabilities.extend(
                probabilities.cpu().numpy()
            )

            all_ids.extend(
                breast_ids
            )


    y_true = np.array(
        all_labels
    )

    y_pred = np.array(
        all_predictions
    )

    y_prob = np.array(
        all_probabilities
    )


    tn, fp, fn, tp = (
        confusion_matrix(
            y_true,
            y_pred
        ).ravel()
    )

    specificity = (
        tn / (tn + fp)
    )


    metrics = {

        "loss":
            total_loss / len(data_loader.dataset),

        "accuracy":
            accuracy_score(
                y_true,
                y_pred
            ),

        "precision":
            precision_score(
                y_true,
                y_pred,
                zero_division=0
            ),

        "recall":
            recall_score(
                y_true,
                y_pred,
                zero_division=0
            ),

        "specificity":
            specificity,

        "f1":
            f1_score(
                y_true,
                y_pred,
                zero_division=0
            ),

        "roc_auc":
            roc_auc_score(
                y_true,
                y_prob
            ),

        "pr_auc":
            average_precision_score(
                y_true,
                y_prob
            )
    }

    prediction_df = pd.DataFrame({

        "patient_breast_id":
            all_ids,

        "target":
            y_true,

        "prediction":
            y_pred,

        "malignant_probability":
            y_prob
    })


    return (
        metrics,
        prediction_df
    )

In [21]:
EPOCHS = 15
PATIENCE = 4

best_val_auc = -1
patience_counter = 0

history = []

BEST_MODEL_FILE = (
    MODEL_DIR
    / "best_two_view_efficientnetv2s.pth"
)


for epoch in range(
    1,
    EPOCHS + 1
):

    model.train()

    running_loss = 0

    for (
        view1,
        view2,
        labels,
        _
    ) in train_loader:

        view1 = view1.to(device)
        view2 = view2.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(
            view1,
            view2
        )

        loss = criterion(
            outputs,
            labels
        )

        loss.backward()

        optimizer.step()

        running_loss += (
            loss.item()
            * labels.size(0)
        )


    train_loss = (
        running_loss
        / len(train_loader.dataset)
    )


    val_metrics, _ = evaluate_model(
        model,
        val_loader
    )


    scheduler.step(
        val_metrics[
            "roc_auc"
        ]
    )


    history.append({

        "epoch":
            epoch,

        "train_loss":
            train_loss,

        "val_loss":
            val_metrics["loss"],

        "val_accuracy":
            val_metrics["accuracy"],

        "val_recall":
            val_metrics["recall"],

        "val_specificity":
            val_metrics["specificity"],

        "val_f1":
            val_metrics["f1"],

        "val_roc_auc":
            val_metrics["roc_auc"],

        "val_pr_auc":
            val_metrics["pr_auc"]
    })


    print(
        f"Epoch {epoch:02d} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Val Loss: {val_metrics['loss']:.4f} | "
        f"Val AUC: {val_metrics['roc_auc']:.4f} | "
        f"Val F1: {val_metrics['f1']:.4f}"
    )


    if (
        val_metrics["roc_auc"]
        > best_val_auc
    ):

        best_val_auc = (
            val_metrics[
                "roc_auc"
            ]
        )

        patience_counter = 0

        torch.save(
            model.state_dict(),
            BEST_MODEL_FILE
        )

        print(
            "  -> Best model saved."
        )

    else:

        patience_counter += 1


    if patience_counter >= PATIENCE:

        print(
            "\nEarly stopping triggered."
        )

        break

Epoch 01 | Train Loss: 0.5575 | Val Loss: 0.5877 | Val AUC: 0.8001 | Val F1: 0.6957
  -> Best model saved.
Epoch 02 | Train Loss: 0.4939 | Val Loss: 0.5462 | Val AUC: 0.8441 | Val F1: 0.7314
  -> Best model saved.
Epoch 03 | Train Loss: 0.4578 | Val Loss: 0.5175 | Val AUC: 0.8195 | Val F1: 0.7796
Epoch 04 | Train Loss: 0.4074 | Val Loss: 0.5428 | Val AUC: 0.8218 | Val F1: 0.7478
Epoch 05 | Train Loss: 0.3509 | Val Loss: 0.4900 | Val AUC: 0.8626 | Val F1: 0.8011
  -> Best model saved.
Epoch 06 | Train Loss: 0.3140 | Val Loss: 0.5302 | Val AUC: 0.8488 | Val F1: 0.7977
Epoch 07 | Train Loss: 0.2680 | Val Loss: 0.8199 | Val AUC: 0.8136 | Val F1: 0.7055
Epoch 08 | Train Loss: 0.2304 | Val Loss: 0.6929 | Val AUC: 0.8483 | Val F1: 0.7516
Epoch 09 | Train Loss: 0.1415 | Val Loss: 0.7454 | Val AUC: 0.8507 | Val F1: 0.7706

Early stopping triggered.


**Save training history**

In [22]:
history_df = pd.DataFrame(
    history
)

history_df.to_csv(
    RESULTS_DIR
    / "efficientnetv2s_training_history.csv",
    index=False
)

history_df

,epoch,train_loss,val_loss,val_accuracy,val_recall,val_specificity,val_f1,val_roc_auc,val_pr_auc
0,1,0.557460,0.587669,0.705502,0.594286,0.850746,0.695652,0.800128,0.840807
1,2,0.493881,0.546241,0.731392,0.645714,0.843284,0.731392,0.844094,0.884002
2,3,0.457770,0.517475,0.734628,0.828571,0.611940,0.779570,0.819488,0.861818
3,4,0.407429,0.542833,0.724919,0.720000,0.731343,0.747774,0.821791,0.857182
4,5,0.350937,0.490045,0.757282,0.862857,0.619403,0.801061,0.862559,0.886548
5,6,0.314033,0.530157,0.770227,0.800000,0.731343,0.797721,0.848785,0.886370
6,7,0.267983,0.819906,0.705502,0.622857,0.813433,0.705502,0.813603,0.857996
7,8,0.230414,0.692856,0.747573,0.674286,0.843284,0.751592,0.848316,0.870578
8,9,0.141548,0.745377,0.757282,0.720000,0.805970,0.770642,0.850704,0.874133


**ConvNeXt two-view model**

In [32]:
from torchvision.models import (
    convnext_tiny,
    ConvNeXt_Tiny_Weights
)

class TwoViewConvNeXt(nn.Module):

    def __init__(self):

        super().__init__()

        weights = ConvNeXt_Tiny_Weights.DEFAULT

        backbone = convnext_tiny(
            weights=weights
        )

        # ConvNeXt-Tiny final feature dimension
        feature_dim = backbone.classifier[2].in_features

        # Keep the normalization + flatten structure,
        # but remove only the final Linear classification layer
        backbone.classifier[2] = nn.Identity()

        self.backbone = backbone

        self.classifier = nn.Sequential(
            nn.Linear(
                feature_dim * 2,
                512
            ),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(
                512,
                2
            )
        )

    def forward(
        self,
        view1,
        view2
    ):

        features1 = self.backbone(view1)
        features2 = self.backbone(view2)

        combined = torch.cat(
            [features1, features2],
            dim=1
        )

        output = self.classifier(
            combined
        )

        return output

**Training setup**

In [33]:
convnext_model = TwoViewConvNeXt().to(device)

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.AdamW(
    convnext_model.parameters(),
    lr=1e-4,
    weight_decay=1e-4
)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="max",
    factor=0.5,
    patience=2
)

BEST_CONVNEXT_FILE = (
    MODEL_DIR
    / "best_two_view_convnext_tiny.pth"
)

In [34]:
view1, view2, labels, breast_ids = next(
    iter(train_loader)
)

view1 = view1.to(device)
view2 = view2.to(device)

with torch.no_grad():

    outputs = convnext_model(
        view1,
        view2
    )

print("Input batch:", view1.shape)
print("Output shape:", outputs.shape)

Input batch: torch.Size([8, 3, 224, 224])
Output shape: torch.Size([8, 2])


**Training loop**

In [35]:
EPOCHS = 15
PATIENCE = 4

best_val_auc = -1
patience_counter = 0

convnext_history = []

for epoch in range(1, EPOCHS + 1):

    convnext_model.train()

    running_loss = 0

    for view1, view2, labels, _ in train_loader:

        view1 = view1.to(device)
        view2 = view2.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = convnext_model(
            view1,
            view2
        )

        loss = criterion(
            outputs,
            labels
        )

        loss.backward()

        optimizer.step()

        running_loss += (
            loss.item()
            * labels.size(0)
        )

    train_loss = (
        running_loss
        / len(train_loader.dataset)
    )

    val_metrics, _ = evaluate_model(
        convnext_model,
        val_loader
    )

    scheduler.step(
        val_metrics["roc_auc"]
    )

    convnext_history.append({
        "epoch": epoch,
        "train_loss": train_loss,
        "val_loss": val_metrics["loss"],
        "val_accuracy": val_metrics["accuracy"],
        "val_recall": val_metrics["recall"],
        "val_specificity": val_metrics["specificity"],
        "val_f1": val_metrics["f1"],
        "val_roc_auc": val_metrics["roc_auc"],
        "val_pr_auc": val_metrics["pr_auc"]
    })

    print(
        f"Epoch {epoch:02d} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Val Loss: {val_metrics['loss']:.4f} | "
        f"Val AUC: {val_metrics['roc_auc']:.4f} | "
        f"Val F1: {val_metrics['f1']:.4f}"
    )

    if val_metrics["roc_auc"] > best_val_auc:

        best_val_auc = val_metrics["roc_auc"]

        patience_counter = 0

        torch.save(
            convnext_model.state_dict(),
            BEST_CONVNEXT_FILE
        )

        print("  -> Best model saved.")

    else:

        patience_counter += 1

    if patience_counter >= PATIENCE:

        print("\nEarly stopping triggered.")

        break

Epoch 01 | Train Loss: 0.6934 | Val Loss: 0.6956 | Val AUC: 0.6803 | Val F1: 0.7231
  -> Best model saved.
Epoch 02 | Train Loss: 0.6850 | Val Loss: 0.6692 | Val AUC: 0.6598 | Val F1: 0.6755
Epoch 03 | Train Loss: 0.6439 | Val Loss: 0.6561 | Val AUC: 0.6529 | Val F1: 0.7300
Epoch 04 | Train Loss: 0.6339 | Val Loss: 0.6506 | Val AUC: 0.6925 | Val F1: 0.5283
  -> Best model saved.
Epoch 05 | Train Loss: 0.5973 | Val Loss: 0.5882 | Val AUC: 0.7558 | Val F1: 0.7532
  -> Best model saved.
Epoch 06 | Train Loss: 0.5598 | Val Loss: 0.5291 | Val AUC: 0.8256 | Val F1: 0.7588
  -> Best model saved.
Epoch 07 | Train Loss: 0.4765 | Val Loss: 0.5072 | Val AUC: 0.8302 | Val F1: 0.7593
  -> Best model saved.
Epoch 08 | Train Loss: 0.4246 | Val Loss: 0.5684 | Val AUC: 0.8219 | Val F1: 0.7508
Epoch 09 | Train Loss: 0.3853 | Val Loss: 0.6751 | Val AUC: 0.8152 | Val F1: 0.7753
Epoch 10 | Train Loss: 0.3107 | Val Loss: 0.5745 | Val AUC: 0.8190 | Val F1: 0.7869
Epoch 11 | Train Loss: 0.2107 | Val Loss: 0.5

**Save ConvNeXt history**

In [36]:
convnext_history_df = pd.DataFrame(
    convnext_history
)

convnext_history_df.to_csv(
    RESULTS_DIR
    / "convnext_tiny_training_history.csv",
    index=False
)

convnext_history_df

,epoch,train_loss,val_loss,val_accuracy,val_recall,val_specificity,val_f1,val_roc_auc,val_pr_auc
0,1,0.693448,0.695557,0.566343,1.000000,0.000000,0.723140,0.680341,0.741864
1,2,0.685049,0.669205,0.605178,0.725714,0.447761,0.675532,0.659829,0.715005
2,3,0.643936,0.656063,0.595469,0.965714,0.111940,0.730022,0.652921,0.708089
3,4,0.633858,0.650554,0.595469,0.400000,0.850746,0.528302,0.692495,0.761761
4,5,0.597296,0.588238,0.692557,0.828571,0.514925,0.753247,0.755821,0.810106
5,6,0.559772,0.529069,0.734628,0.737143,0.731343,0.758824,0.825629,0.859269
6,7,0.476468,0.507187,0.747573,0.702857,0.805970,0.759259,0.830235,0.871736
7,8,0.424616,0.568367,0.744337,0.680000,0.828358,0.750789,0.821919,0.862827
8,9,0.385271,0.675091,0.705502,0.897143,0.455224,0.775309,0.815224,0.851956
9,10,0.310717,0.574533,0.747573,0.822857,0.649254,0.786885,0.819019,0.860557


**evaluate the untouched test set once**

In [37]:
# ============================================================
# FINAL IMAGE MODEL TEST EVALUATION
# Selected model: ConvNeXt-Tiny
# Selection criterion: Validation ROC-AUC
# ============================================================

# Recreate model architecture
final_image_model = TwoViewConvNeXt().to(device)

# Load BEST validation checkpoint
final_image_model.load_state_dict(
    torch.load(
        BEST_CONVNEXT_FILE,
        map_location=device
    )
)

final_image_model.eval()

print("Loaded checkpoint:")
print(BEST_CONVNEXT_FILE)

# Evaluate on untouched test set
test_metrics, test_predictions = evaluate_model(
    final_image_model,
    test_loader
)

print("\n" + "=" * 70)
print("FINAL IMAGE MODEL TEST RESULTS")
print("=" * 70)

for key, value in test_metrics.items():

    print(
        f"{key}: {value:.4f}"
    )

# Save metrics
test_metrics_df = pd.DataFrame(
    [
        {
            "model": "ConvNeXt-Tiny",
            "architecture": "Two-view shared backbone",
            "selected_epoch": 13,
            "selection_metric": "validation_roc_auc",
            "best_validation_roc_auc": 0.863539,
            **test_metrics
        }
    ]
)

test_metrics_df.to_csv(
    RESULTS_DIR
    / "final_image_test_metrics.csv",
    index=False
)

# Add identifying information
test_predictions = test_predictions.merge(
    test_df[
        [
            "patient_breast_id",
            "patient_id",
            "breast_side",
            "classification",
            "split"
        ]
    ],
    on="patient_breast_id",
    how="left",
    validate="one_to_one"
)

test_predictions.to_csv(
    RESULTS_DIR
    / "final_image_test_predictions.csv",
    index=False
)

print("\nSaved:")
print("- final_image_test_metrics.csv")
print("- final_image_test_predictions.csv")

Loaded checkpoint:
/content/drive/MyDrive/Dissertation/04_Models/saved_models/best_two_view_convnext_tiny.pth

FINAL IMAGE MODEL TEST RESULTS
loss: 0.9139
accuracy: 0.7413
precision: 0.8243
recall: 0.6854
specificity: 0.8129
f1: 0.7485
roc_auc: 0.8284
pr_auc: 0.8628

Saved:
- final_image_test_metrics.csv
- final_image_test_predictions.csv


**Save a validation comparison table**

In [38]:
image_model_comparison = pd.DataFrame([
    {
        "model": "EfficientNetV2-S",
        "best_epoch": 5,
        "val_accuracy": 0.757282,
        "val_recall": 0.862857,
        "val_specificity": 0.619403,
        "val_f1": 0.801061,
        "val_roc_auc": 0.862559,
        "val_pr_auc": 0.886548
    },
    {
        "model": "ConvNeXt-Tiny",
        "best_epoch": 13,
        "val_accuracy": 0.773463,
        "val_recall": 0.737143,
        "val_specificity": 0.820896,
        "val_f1": 0.786585,
        "val_roc_auc": 0.863539,
        "val_pr_auc": 0.888012
    }
])

image_model_comparison.to_csv(
    RESULTS_DIR
    / "image_model_validation_comparison.csv",
    index=False
)

image_model_comparison

,model,best_epoch,val_accuracy,val_recall,val_specificity,val_f1,val_roc_auc,val_pr_auc
0,EfficientNetV2-S,5,0.757282,0.862857,0.619403,0.801061,0.862559,0.886548
1,ConvNeXt-Tiny,13,0.773463,0.737143,0.820896,0.786585,0.863539,0.888012
